In [57]:
from dataset import DartsDataset
import os
import json
from torch.utils.data import random_split
import torch

In [58]:
dirs = [f'3D/scene/rendered/imgs_{i}' for i in range(9)]
dataset = DartsDataset(dirs=dirs, random_rotation=False, random_rescale=False)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, validation_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
test_dataset = DartsDataset(dirs=[f'my_unlabelled/vids/frames_000{i}' for i in range(8)], random_rescale=False, random_rotation=False, is_synthetic=False)

In [59]:
categories = [
    {
        "id": 0,
        "name": "right",
        "supercategory": "corner",
    },
    {
        "id": 1,
        "name": "top",
        "supercategory": "corner",
    },
    {
        "id": 2,
        "name": "left",
        "supercategory": "corner",
    },
    {
        "id": 3,
        "name": "bottom",
        "supercategory": "corner",
    },
    {
        "id": 4,
        "name": "dart",
        "supercategory": "dart",
    }
]

In [60]:
def make_coco_dataset(dataset, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    images = []
    annotations = []
    anno_id = 0
    for idx, (path, img, anno1, img_transformed, anno2) in enumerate(dataset):
        filename = os.path.basename(path)

        img.save(f"{output_dir}/{filename}")
        img_transformed.save(f"{output_dir}/transformed_{filename}")
        images.append({
            "id": 2*idx,
            "width": img.width,
            "height": img.height,
            "file_name": filename,
            "date_captured": "",
        })
        images.append({
            "id": 2*idx+1,
            "width": img_transformed.width,
            "height": img_transformed.height,
            "file_name": f"transformed_{filename}",
            "date_captured": "",
        })
        
        for anno in anno1["annotations"]:
            anno["id"] = anno_id
            anno["image_id"] = 2*idx
            anno["bbox"] = list(map(float, anno["bbox"].numpy()))
            anno["category_id"] = int(anno["category_id"])
            anno["area"] = float(anno["area"])
            anno_id += 1
            annotations.append(anno)


        for anno in anno2["annotations"]:
            anno["id"] = anno_id
            anno["image_id"] = 2*idx+1
            anno["bbox"] = list(map(float, anno["bbox"].numpy()))
            anno["category_id"] = int(anno["category_id"])
            anno["area"] = float(anno["area"])
            anno_id += 1
            annotations.append(anno)

    dataset_dict = {
        "info": {
            "description": "Darts Dataset"
        },
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }

    with open(f"{output_dir}/_annotations.coco.json", "w") as f:
        json.dump(dataset_dict, f)

In [61]:
make_coco_dataset(train_dataset, "dataset/train")
make_coco_dataset(validation_dataset, "dataset/valid")
make_coco_dataset(test_dataset, "dataset/test")